<a href="https://colab.research.google.com/github/hsy0828/fcnv2-weather-demo/blob/main/run_in_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

GITHUB_USER = "hsy0828"
REPO_NAME = "fcnv2-weather-demo"
REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
TARGET_DIR = f"/content/{REPO_NAME}"

if not os.path.exists(TARGET_DIR):
    !git clone {REPO_URL} {TARGET_DIR}
else:
    %cd {TARGET_DIR}
    !git pull

%cd {TARGET_DIR}

print("=== 1. 安裝系統級庫 ===")
!apt-get update -qq && apt-get install -y -qq libeccodes-dev

print("=== 2. 強制安裝 Wheel 預編譯套件 ===")
!pip install --only-binary=:all: ai-models ai-models-fourcastnetv2-gfs earthkit-meteo eccodes xarray cfgrib matplotlib cartopy onnxruntime onnx || pip install ai-models ai-models-fourcastnetv2-gfs earthkit-meteo eccodes xarray cfgrib matplotlib cartopy onnxruntime onnx

print("=== 3. 降級鎖定 earthkit-data ===")
!pip install "earthkit-data==0.9.4" --force-reinstall --no-deps

print("=== 安裝完成！請接著執行下一個 Cell ===")

In [ ]:
%cd /content/fcnv2-weather-demo

# 測試用 Python 直接啟動預測
!python -c "import sys, ai_models_fourcastnetv2_gfs; from ai_models.model import MODELS; from ai_models_fourcastnetv2_gfs.model import FourCastNetv2SmallModel; MODELS['fourcastnetv2-small'] = FourCastNetv2SmallModel; from ai_models.__main__ import main; sys.argv=['ai-models', '--input', 'ecmwf-open-data', '--date', '20240101', '--time', '0000', '--lead-time', '6', '--lead-time', '12', '--lead-time', '18', '--lead-time', '24', '--lead-time', '30', '--lead-time', '36', '--lead-time', '42', '--lead-time', '48', '--lead-time', '54', '--lead-time', '60', '--lead-time', '66', '--lead-time', '72', 'fourcastnetv2-small']; main()"

In [ ]:
%cd /content/fcnv2-weather-demo

print("=== 檢查已註冊的模型清單 ===")
!ai-models --models

print("\n=== 開始預測 ===")
!python predict.py

In [ ]:
import sys

print("=== 診斷 1: 檢查套件是否安裝在 site-packages ===")
!pip list | grep -E "ai-models|earthkit"

print("\n=== 診斷 2: 強制 Import 插件模組 ===")
try:
    import ai_models_fourcastnetv2_gfs

    print("SUCCESS: 插件模組已成功 import！")
except Exception as e:
    print(f"FAILED: 匯入插件時發生錯誤！詳細訊息如下：")
    import traceback

    traceback.print_exc()

In [ ]:
from IPython.display import Image, display

# 顯示繪製完成的預測地圖
display(Image('forecast_temp_wind.png'))